In [2]:
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import precision_recall_curve, average_precision_score
from src.data_loader import temporal_split, stratified_sample
from src.text_processing import build_tfidf_pipeline, compute_text_features, compute_readability
from src.feature_engineering import compute_metadata_features, build_feature_matrix
from src.models import compare_models

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")

df = pd.read_csv(config.DATA_PROCESSED / "reviews_clean.csv")
df["DateTime"] = pd.to_datetime(df["Time"], unit="s")

# Filtrage : uniquement les avis ayant reçu >= 3 votes[cite: 4, 6]
df_eval = df[df["HelpfulnessDenominator"] >= config.MIN_VOTES].copy()
sample = stratified_sample(df_eval, n=25000, stratify_col="IsHelpful", random_state=config.RANDOM_STATE)
train_df, test_df = temporal_split(sample, config.TRAIN_END_YEAR, config.TEST_START_YEAR)

print(f"Train : {len(train_df):,} avis | Test : {len(test_df):,} avis")
print(f"Taux d'avis utiles (Train) : {train_df['IsHelpful'].mean()*100:.1f}%")

Train : 15,104 avis | Test : 9,896 avis
Taux d'avis utiles (Train) : 70.1%


In [ ]:
# 1. TF-IDF + SVD
tfidf_svd = build_tfidf_pipeline(max_features=3000, n_components=30)
X_tfidf_tr = tfidf_svd.fit_transform(train_df["Text"].fillna(""))
X_tfidf_te = tfidf_svd.transform(test_df["Text"].fillna(""))

# 2. Features stylistiques + lisibilité
tr_text = compute_text_features(train_df)
te_text = compute_text_features(test_df)
tr_text["readability"] = compute_readability(train_df["Text"])
te_text["readability"] = compute_readability(test_df["Text"])

# 3. Métadonnées anti-fuite
tr_meta, prod_means = compute_metadata_features(train_df)
te_meta, _ = compute_metadata_features(test_df, train_product_means=prod_means)

y_train = train_df["IsHelpful"].values
y_test = test_df["IsHelpful"].values

# Étude d'ablation avec GradientBoosting
from sklearn.ensemble import GradientBoostingClassifier

ablations = {
    "1. Métadonnées seules (Note, Produit)": build_feature_matrix(train_df, None, None, None, tr_meta)[0],
    "2. Texte TF-IDF + Style/Lisibilité": build_feature_matrix(train_df, X_tfidf_tr, None, tr_text, None)[0],
    "3. Complet (TF-IDF + Style + Métadonnées)": build_feature_matrix(train_df, X_tfidf_tr, None, tr_text, tr_meta)[0],
}
ablations_test = {
    "1. Métadonnées seules (Note, Produit)": build_feature_matrix(test_df, None, None, None, te_meta)[0],
    "2. Texte TF-IDF + Style/Lisibilité": build_feature_matrix(test_df, X_tfidf_te, None, te_text, None)[0],
    "3. Complet (TF-IDF + Style + Métadonnées)": build_feature_matrix(test_df, X_tfidf_te, None, te_text, te_meta)[0],
}

ablation_rows = []
for label in ablations:
    clf = GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
    clf.fit(ablations[label], y_train)
    proba = clf.predict_proba(ablations_test[label])[:, 1]
    ablation_rows.append({"Configuration": label, "PR-AUC": round(average_precision_score(y_test, proba), 4)})

display(pd.DataFrame(ablation_rows))

TypeError: build_tfidf_pipeline() got an unexpected keyword argument 'random_state'

In [ ]:
X_train, feat_names = build_feature_matrix(train_df, X_tfidf_tr, None, tr_text, tr_meta)
X_test, _ = build_feature_matrix(test_df, X_tfidf_te, None, te_text, te_meta)

results_df, trained_models = compare_models(X_train, y_train, X_test, y_test)
display(results_df)

# Tracé et sauvegarde des courbes Precision-Recall
fig, ax = plt.subplots(figsize=(10, 6))
for name, model in trained_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax.plot(rec, prec, label=f"{name} (PR-AUC = {ap:.3f})")

ax.axhline(y=y_test.mean(), color="gray", linestyle="--", label=f"Baseline aléatoire ({y_test.mean():.2f})")
ax.set_xlabel("Recall (Rappel)")
ax.set_ylabel("Precision (Précision)")
ax.set_title("Courbes Precision-Recall — Prédiction d'utilité (Test >= 2011)", fontweight="bold")
ax.legend(loc="best")
plt.tight_layout()
plt.savefig(str(config.FIGURES_DIR / "pr_curves_utilite.png"), dpi=150)
plt.show()

In [ ]:
rf_model = trained_models["RandomForest"]
importances = pd.Series(rf_model.feature_importances_, index=feat_names).nlargest(15)

fig, ax = plt.subplots(figsize=(10, 6))
importances.sort_values().plot(kind="barh", ax=ax, color="#2b8cbe", edgecolor="black")
ax.set_title("Top 15 des variables les plus prédictives de l'utilité", fontweight="bold")
ax.set_xlabel("Importance (Gini)")
plt.tight_layout()
plt.savefig(str(config.FIGURES_DIR / "feature_importance_utilite.png"), dpi=150)
plt.show()
